# Four Formats, One Analysis

You are about to run one real financial analysis on data that arrives in **four different file formats** — because that is how data actually shows up.

| File | Format | Source | Why that source uses that format |
|---|---|---|---|
| `DGS10.csv` | CSV | FRED (St. Louis Fed) | maximum compatibility — everything opens CSV |
| `portfolio.json` | JSON | a broker-style API | holdings *contain* trades — nested, not tabular |
| `fred_macro.xlsx` | Excel | FRED's export button | made for humans: multiple sheets, title rows |
| `stock_market.parquet` | Parquet | a data-pipeline world | typed, compressed, columnar — made for machines |

**The theory table** — you will verify every row of it with your own hands:

| Format | Human-readable? | Nested data? | Dtypes survive? | Size | Read speed | Where you'll experience it |
|---|---|---|---|---|---|---|
| CSV | ✅ | ❌ | ❌ everything is text | medium | medium | Task 1.1 (the `object` surprise), Task 4 |
| JSON | ✅ | ✅ | ❌ | large (keys repeat per record) | slow | Task 1.2 (nesting), Task 4 |
| Excel | via app | sheets, sort of | partially | large | slowest | Task 1.3 (metadata rows), Task 4 |
| Parquet | ❌ binary | ✅ | ✅ | smallest | fastest (columnar) | Task 1.4 (dtypes + column reads), Task 4 |

**How grading works:** after each task, run the ✅ cell below it — it prints pass/fail with a pointer (never the answer). At the end, `grader.summary()` shows your score. Checks never crash your notebook, and re-running a check overwrites its result, so iterate freely. Stuck? Open `hints.md`.

**Estimated time:** 2.5–3.5 hours.

> ⚠️ *Loading a file is step one; trusting it is a mistake.*


In [ ]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

assert Path("utils/test.py").exists(), "Run this notebook from the repo root (pandas-finance-assignment/)"

# autoreload: when you save an edit to utils/finance.py, the notebook re-imports it
# automatically — no kernel restart needed. That's part of the .py-module workflow.
%load_ext autoreload
%autoreload 2

from utils import data_setup
from utils import test as grader

Path("data").mkdir(exist_ok=True)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)
print("Setup complete — pandas", pd.__version__)

## Part 0 — Get the data

Data acquisition **is** part of analysis: whoever publishes the data picks the format, and the format encodes *who the data is for*. FRED serves CSV because every tool on earth opens it. Broker APIs speak JSON because a portfolio isn't a table — holdings *contain* trades. Humans exchange Excel because it carries formatting, notes and multiple sheets. Data pipelines use Parquet because at scale, types, compression and column-reads are what matter.

The download logic lives in `utils/data_setup.py` (go read it later — it's short). Each cell below fetches one file into `data/`.

In [ ]:
# 0.1 — CSV from FRED: the 10-year US Treasury yield, daily since 2021.
data_setup.download_fred_csv()

In [ ]:
# 0.2 — JSON: a broker-style portfolio (deterministic, written locally).
data_setup.write_portfolio_json()

# Look at the raw file before pandas touches it — see the nesting?
print("".join(Path("data/portfolio.json").read_text().splitlines(keepends=True)[:15]))

In [ ]:
# 0.3 — Excel: CPI + unemployment, in FRED's human-facing export style.
data_setup.download_fred_excel()

In [ ]:
# 0.4 — Parquet: daily OHLCV for AAPL, MSFT, SPY (via yfinance), tidied.
data_setup.download_stocks_parquet()

### 📴 Offline fallback

No network, or a download failed? The repo ships dated **real-data snapshots** from the same FRED and Yahoo Finance acquisition paths in `data_offline/`. Uncomment the next cell and run it — every task and every check works the same. (One difference: the offline stock file already has the deterministic teaching defects, so you'll **skip** the chaos cell in Part 2 — it will tell you so itself.) The Yahoo-derived snapshot is private-course material and must not be redistributed.

In [ ]:
# data_setup.use_offline_data()   # ← uncomment and run only if a download above failed

In [ ]:
grader.check_0_data_files()

## Part 1 — Loading: each file teaches one row of the theory table

Same `pd.read_*` family, four very different experiences. For each format, watch **what pandas gets right on its own** and **what you must tell it**. After each task, fill in the one-sentence takeaway.

In [ ]:
# 1.1 — CSV: the naive load. No arguments, no safety net.
naive = pd.read_csv("data/DGS10.csv")
print(naive.dtypes)      # why is DGS10 an 'object' (strings!) and not a number?
naive.head()

# Something in the raw file forces the whole column to text. Open data/DGS10.csv
# in a text editor and look closely at the value column. Then:
#
# TODO: reload into `rates` so that
#   * the culprit values become proper NaN  (read_csv has an argument for this)
#   * DATE is parsed as real dates at load time
rates = ...

# TODO: print how many values in DGS10 are missing (NaN).

In [ ]:
grader.check_1_1(rates)

**Format takeaway (1 sentence):** ______  *(what did CSV forget about your data, and what did you have to tell pandas?)*

In [ ]:
# 1.2 — JSON: not a table. A structure.
with open("data/portfolio.json") as f:
    p = json.load(f)

print(type(p))                     # a dict, not a DataFrame
print(list(p.keys()))
p["holdings"][0]["trades"]         # a list, inside a dict, inside a list

# TODO: flatten the nesting with pd.json_normalize — build `trades` with ONE ROW
# PER TRADE (4 rows), carrying each holding's ticker and share count along with a
# "holding_" prefix. The docs for record_path= and meta= are your map.
trades = ...
trades

In [ ]:
grader.check_1_2(trades)

*Why can't CSV represent `portfolio.json` without either duplicating the owner data on every row or splitting it into multiple files?*

**Format takeaway (1 sentence):** ______

In [ ]:
# 1.3 — Excel: a workbook, not a file. And it was formatted for eyes, not parsers.
xl = pd.ExcelFile("data/fred_macro.xlsx")
print(xl.sheet_names)              # more than one sheet!

naive = pd.read_excel("data/fred_macro.xlsx")
naive.head()                       # look at the "column names" — that's metadata, not data

# TODO: reload into `macro`: the "Monthly" sheet, with the decorative rows skipped
# so the real header is used, and observation_date as a datetime index.
macro = ...


In [ ]:
grader.check_1_3(macro)

*Those metadata rows genuinely help a human reader. What does that tell you about who Excel files are made for — and what does it cost the parser?*

**Format takeaway (1 sentence):** ______

In [ ]:
# 1.4 — Parquet: the format that remembers.
stocks = pd.read_parquet("data/stock_market.parquet")
print(stocks.dtypes)               # Date is ALREADY datetime64 — compare with 1.1!
print(sorted(stocks["Ticker"].unique()))   # hmm... remember this for Part 2.

# Columnar superpower: Parquet can read a subset of COLUMNS without touching the rest.
%time _full = pd.read_parquet("data/stock_market.parquet")

# TODO: time a second read that loads ONLY Date, Ticker and Close
# (read_parquet has an argument for exactly this).


In [ ]:
grader.check_1_4(stocks)

*Why is a column subset fast in Parquet, but impossible to shortcut in CSV? (Think about how each file is laid out on disk.)*

**Format takeaway (1 sentence):** ______

### Part 1 checkpoint — fill in what you observed

| Format | Dtypes survived? | Extra load arguments needed? | Flat or nested? |
|---|---|---|---|
| CSV | | | |
| JSON | | | |
| Excel | | | |
| Parquet | | | |

## Part 2 — Cleaning: the data fights back

### 2.0 ⚠️ Chaos cell (recommended)

Your Parquet file is suspiciously pristine. Real market data that has passed through years of pipelines is not. The next cell **deliberately injects** the four classic problems — duplicated rows, inconsistent ticker casing, missing prices, and one fat-finger price — exactly the way real glitches do.

Run it once. (**Offline users:** your file is already messed up — the cell will detect that and skip itself. Skipping entirely is allowed too, but the cleaning checks will note they couldn't verify the mess existed.)

In [ ]:
try:
    stocks = data_setup.inject_chaos(stocks)
except RuntimeError as e:
    print(f"Not injected: {e}")

In [ ]:
# 2.1 — Clean the stock table. Order matters — think about why as you go.
stocks_clean = stocks.copy()

# TODO (1) normalize Ticker to uppercase — 'aapl' and 'AAPL' are the same company.

# TODO (2) drop duplicate rows: a ticker gets ONE row per date.
#          (drop_duplicates has a subset= argument.)

# TODO (3) sort by Ticker then Date and reset the index — shift() and ffill()
#          silently assume ordered data.

# TODO (4) the fat-finger outlier: WITHIN each ticker, divide each Close by the
#          previous day's Close (groupby("Ticker")["Close"].shift() gives you the
#          previous one). A ratio above 5 or below 0.2 is a typo, not a market —
#          set those Close values to NaN.

# TODO (5) fill every missing Close with that ticker's previous Close:
#          groupby("Ticker")["Close"].ffill()

stocks_clean.head()

*Why would filling missing prices with the column MEAN be catastrophic here, when it's a reasonable default elsewhere? (What would a 2021 gap in AAPL get filled with?)*

In [ ]:
grader.check_2_1(stocks_clean)

In [ ]:
# 2.2 — Clean the other two tables. Match the fix to the mess:
#   rates: '.' markers became NaN   ← mess origin: CSV stores text, not types (Task 1.1)
#   macro: decorative layout, gaps  ← mess origin: Excel is made for humans (Task 1.3)

# TODO: rates_clean — DATE as a sorted index, then forward-fill DGS10
# (markets were closed those days: carry yesterday's yield).
rates_clean = ...

# TODO: macro_clean — copy macro, fill UNRATE's gaps with .interpolate()
# (a slow, smooth series: a straight line between neighbors is honest).
macro_clean = ...

*Three repair tools, three philosophies — when is each one right?*
- `ffill` — "nothing changed while we weren't looking" *(prices, yields on holidays)*
- `interpolate` — "it moved smoothly between the points we have" *(slow macro indicators)*
- `dropna` — "this row is beyond saving" *(when rows are plentiful and independence matters)*

In [ ]:
grader.check_2_2(rates_clean, macro_clean)

## Part 3 — Finance: the payoff

### From notebook to module

So far your logic lives in cells. Real projects keep reusable logic in **`.py` modules** instead: modules can be imported by other notebooks and scripts, tested automatically, and reviewed line-by-line in version control — notebook cells can't. In this part you will implement five functions in **`utils/finance.py`**. The grader tests them on tiny synthetic datasets **with known answers**, and only once they're green will you point them at real market data.

Remember the setup cell loaded `%autoreload`: **save the .py file and just rerun the check cell** — no kernel restart.

In [ ]:
# 3.1 — Warm-up (inline, last one before the module).
# TODO: reshape stocks_clean into a WIDE table `close`: one row per Date, one
# column per Ticker, values = Close. (One method call. Long -> wide.)
close = ...

# TODO: daily simple returns for every ticker (one call on `close`).
daily_ret = ...

# TODO: total compounded return per ticker over the whole sample:
# grow $1 through every day's return, i.e. (1 + r).prod() - 1. NOT the sum!
cum_ret = ...

cum_ret  # which ticker won?

In [ ]:
grader.check_3_1(close, daily_ret, cum_ret)

### 3.2 — Implement the module

Open **`utils/finance.py`** in an editor. Five functions are waiting with full docstrings (formula, input/output types, NaN behavior — the docstring fully determines the answer):

`daily_returns` · `cumulative_return` · `annualized_volatility` · `sharpe_ratio` · `max_drawdown`

Implement them, **save the file**, then run the check cell below. It prints one line per function — iterate until all five are green. (You already wrote two of them inline in 3.1 — just generalize.)

In [ ]:
grader.check_finance_module()

In [ ]:
# 3.3 — Point YOUR module at real market data.
from utils.finance import annualized_volatility, cumulative_return, max_drawdown, sharpe_ratio

# TODO: annualized volatility per ticker — your module function, on daily_ret.
vol = ...

# TODO: the risk-free rate = the AVERAGE 10y Treasury yield over the sample.
# Careful: DGS10 is in percent (3.5 means 3.5%); your functions want a decimal float.
rf = ...

# TODO: Sharpe ratio per ticker (your module function again).
sharpe = ...

# TODO: AAPL 20-day and 50-day moving averages of the close (rolling means).
ma20 = ...
ma50 = ...

# Optional plot — uncomment once ma20/ma50 are real:
# import matplotlib.pyplot as plt
# ax = close["AAPL"].plot(figsize=(10, 4), label="AAPL close", alpha=0.5)
# ma20.plot(ax=ax, label="20-day MA"); ma50.plot(ax=ax, label="50-day MA")
# ax.legend(); ax.set_title("AAPL — price vs moving averages"); plt.show()

print(vol, rf, sharpe, sep="\n")

*Interpretation checkpoints:* SPY should show the **lowest** volatility — it holds 500 stocks, and diversification cancels idiosyncratic noise. And a Sharpe ratio above **1** is the classic "paid fairly for the risk" rule of thumb — who clears it?

In [ ]:
grader.check_3_3(vol, sharpe, rf)

In [ ]:
# 3.4 — Max drawdown: the deepest peak-to-trough fall (the pain metric).
# TODO: apply your max_drawdown to EACH COLUMN of `close` (DataFrames have a
# method that applies a function per column), then report it as a percentage.
mdd = ...
mdd

In [ ]:
grader.check_3_4(mdd, close)

In [ ]:
# 3.5 — Portfolio valuation: the JSON meets the Parquet.
# The payoff of Part 1: hierarchy (holdings/trades) -> flat table (1.2) -> joined
# with market prices (1.4). What you PAID lives in the trades; what it's WORTH
# lives in the latest close.

# TODO: build `portfolio_df` — one row per holding, columns exactly:
#   ticker        the holding's ticker
#   shares        holding-level share count
#   cost_basis    what was paid: sum over that holding's trades of shares x price
#                 (you flattened the trades in 1.2 — group them!)
#   market_value  what it's worth now: holding shares x the most recent close
#                 (where in your `close` table do the LATEST prices live?)
#   pnl           market_value - cost_basis
#   pnl_pct       pnl relative to cost_basis
# Also print the portfolio totals (invested, current value, total P&L).
portfolio_df = ...
portfolio_df

In [ ]:
grader.check_3_5(portfolio_df, close)

In [ ]:
# 3.6 — Macro link: the Excel meets the Parquet.
# Does the S&P's monthly return co-move with CHANGES in unemployment?

# TODO: SPY monthly returns — take the FIRST close of each month, then pct_change.
# (.resample("MS") groups a datetime index by Month Start.)
spy_monthly = ...

# TODO: month-over-month CHANGE in UNRATE (level minus previous level).
unrate_change = ...

# TODO: correlate the two into a single float `corr`. (Series.corr aligns the
# two series on their shared monthly index for you.)
corr = ...
print(corr)

*One-sentence interpretation: what sign did you get, and does it make economic sense?*  ______

**Honesty note:** this is ~60 monthly observations. That's an illustration, not evidence — a real study would want decades of data, controls, and out-of-sample checks. Treat the number as a conversation starter.

In [ ]:
grader.check_3_6(corr)

## Part 4 — Capstone: rebuild the comparison table from evidence

You've *felt* each format's personality. Now measure it. Write `stocks_clean` to all four formats; time the write, time the read, record the file size, and test whether the `Date` dtype survives the round trip. **No trusting the theory table — build your own.**

In [ ]:
# The writer/reader table is given — your job is the measuring.
# (read_json(convert_dates=False) stops pandas from GUESSING dates from column
#  names — we want to know what the FORMAT itself preserves. read_csv gets the
#  same deal by default: no arguments, no favors.)
io_funcs = {
    "csv": (lambda df, p: df.to_csv(p, index=False),
            lambda p: pd.read_csv(p)),
    "json": (lambda df, p: df.to_json(p, orient="records", date_format="iso"),
             lambda p: pd.read_json(p, convert_dates=False)),
    "xlsx": (lambda df, p: df.to_excel(p, index=False),
             lambda p: pd.read_excel(p)),
    "parquet": (lambda df, p: df.to_parquet(p, index=False),
                lambda p: pd.read_parquet(p)),
}

rows = []
for fmt, (write_fn, read_fn) in io_funcs.items():
    path = Path("data") / f"benchmark.{fmt}"

    # TODO: time the write — time.perf_counter() before and after
    #       write_fn(stocks_clean, path)
    write_s = ...

    # TODO: time the read the same way; keep the loaded result as `reloaded`.
    read_s = ...
    reloaded = ...

    # TODO: file size in KB — os.path.getsize gives bytes.
    size_kb = ...

    # TODO: did the Date dtype survive the round trip? Use
    # pd.api.types.is_datetime64_any_dtype(reloaded["Date"])   (any resolution counts)
    survives = ...

    rows.append({"format": fmt, "write_s": write_s, "read_s": read_s,
                 "size_kb": size_kb, "date_dtype_survives": survives})

results_df = pd.DataFrame(rows)
results_df

In [ ]:
grader.check_4(results_df)

### Reflection — theory vs your measurements

Put your `results_df` next to the theory table in Cell 1. Expected patterns worth naming (do your numbers show them?):
- **JSON is the largest** — every record repeats every key.
- **Excel is the slowest** — it's a zipped XML workbook wearing a spreadsheet costume. (Did its `Date` dtype survive? Excel stores real date *cells*, so it often does — a nuance the simple theory table misses.)
- **Parquet is the smallest and fastest, and the only one whose types are guaranteed** by the format itself.

Now make four calls — and justify each with **a row of your own numbers**, not vibes:

1. **Email a table to a manager** → format? because…
2. **Store 10 years of tick data** → format? because…
3. **Design an API response** → format? because…
4. **Quick one-off share with a colleague who "just wants to open it"** → format? because…

In [ ]:
grader.summary()

### ✅ Before you submit — self-check

- [ ] Part 0: all four files downloaded (or offline fallback used)
- [ ] Part 1: all four loads pass, **and I wrote a format takeaway for all four formats**
- [ ] Part 1 checkpoint table filled in
- [ ] Part 2: chaos handled — cleaning checks pass
- [ ] Part 3: `utils/finance.py` fully implemented (module check green) and all analysis checks pass
- [ ] Part 4: benchmark table built from my own measurements; reflection scenarios answered
- [ ] `grader.summary()` shows my final score
